# Notebook 1 - ETL & Data Engineering

**Project phase:** Formula 1 Big Data Analysis with PySpark  
**Spark components:** RDDs, DataFrames, SparkSQL  
**Goal:** Ingest the 14 raw Formula 1 CSV files into a medallion architecture and create clean analytical-ready Gold tables.

The dataset is compiled from the Ergast Formula 1 database and covers races, drivers, constructors, circuits, qualifying, lap times, pit stops, standings, sprint results, and race statuses from 1950 through 2024.

## Big-Data Engineering Principles Used

- Explicit schemas are used for all CSV ingestion. We avoid `inferSchema=True` because schema inference requires extra scans and can be unsafe at scale.
- Parquet is used for lakehouse storage because it is columnar, compressed, and efficient for analytical queries.
- Gold tables are partitioned by `year` where applicable to improve pruning for time-based analysis.
- Small dimension tables are joined with `broadcast()` where suitable.
- Python-native `sorted()` and pre-aggregation `.collect()` are avoided. Ordering and aggregation are handled by Spark.
- RDD logic is included for pedagogical completeness, but DataFrames and SparkSQL are preferred for production because Catalyst and Tungsten optimize query planning and execution.

## Google Colab Setup

Run this cell first in Google Colab. It installs the required libraries and mounts Google Drive. The raw CSV files are expected to be available at `DATASET_PATH = "/content/drive/MyDrive/Dataset"`.

In [5]:
!pip install -q pyspark pandas numpy matplotlib seaborn pyarrow

from google.colab import drive
drive.mount('/content/drive')

DATASET_PATH = "/content/drive/MyDrive/data_analysis"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
from pyspark.sql.functions import broadcast

spark = (
    SparkSession.builder
    .appName("F1 Notebook 1 - ETL and Data Engineering")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark

In [7]:
RAW_PATH = DATASET_PATH
BRONZE_PATH = "/content/lakehouse/bronze"
SILVER_PATH = "/content/lakehouse/silver"
GOLD_PATH = "/content/lakehouse/gold"

if not os.path.exists(RAW_PATH):
    raise FileNotFoundError(f"Dataset folder not found: {RAW_PATH}. Mount Google Drive and confirm the CSV files are in this folder.")

CSV_OPTIONS = {
    "header": "true",
    "quote": '"',
    "escape": '"',
    "nullValue": "\\N",
    "mode": "PERMISSIVE"
}

def read_csv_with_schema(file_name: str, schema: T.StructType):
    return (
        spark.read
        .options(**CSV_OPTIONS)
        .schema(schema)
        .csv(f"{RAW_PATH}/{file_name}")
        .withColumn("ingested_at", F.current_timestamp())
        .withColumn("source_file", F.lit(file_name))
    )

def write_parquet(df, path: str, partition_cols=None):
    writer = df.write.mode("overwrite").format("parquet")
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.save(path)

## Bronze Layer - Explicit Schema Ingestion

The Bronze layer keeps the raw business meaning of the source files while converting them to Parquet and adding audit columns. These tables are intentionally close to the source so downstream transformations remain traceable.

In [8]:
schemas = {
    "circuits.csv": T.StructType([
        T.StructField("circuitId", T.IntegerType()), T.StructField("circuitRef", T.StringType()),
        T.StructField("name", T.StringType()), T.StructField("location", T.StringType()),
        T.StructField("country", T.StringType()), T.StructField("lat", T.DoubleType()),
        T.StructField("lng", T.DoubleType()), T.StructField("alt", T.IntegerType()),
        T.StructField("url", T.StringType())]),
    "constructor_results.csv": T.StructType([
        T.StructField("constructorResultsId", T.IntegerType()), T.StructField("raceId", T.IntegerType()),
        T.StructField("constructorId", T.IntegerType()), T.StructField("points", T.DoubleType()),
        T.StructField("status", T.StringType())]),
    "constructor_standings.csv": T.StructType([
        T.StructField("constructorStandingsId", T.IntegerType()), T.StructField("raceId", T.IntegerType()),
        T.StructField("constructorId", T.IntegerType()), T.StructField("points", T.DoubleType()),
        T.StructField("position", T.IntegerType()), T.StructField("positionText", T.StringType()),
        T.StructField("wins", T.IntegerType())]),
    "constructors.csv": T.StructType([
        T.StructField("constructorId", T.IntegerType()), T.StructField("constructorRef", T.StringType()),
        T.StructField("name", T.StringType()), T.StructField("nationality", T.StringType()),
        T.StructField("url", T.StringType())]),
    "driver_standings.csv": T.StructType([
        T.StructField("driverStandingsId", T.IntegerType()), T.StructField("raceId", T.IntegerType()),
        T.StructField("driverId", T.IntegerType()), T.StructField("points", T.DoubleType()),
        T.StructField("position", T.IntegerType()), T.StructField("positionText", T.StringType()),
        T.StructField("wins", T.IntegerType())]),
    "drivers.csv": T.StructType([
        T.StructField("driverId", T.IntegerType()), T.StructField("driverRef", T.StringType()),
        T.StructField("number", T.IntegerType()), T.StructField("code", T.StringType()),
        T.StructField("forename", T.StringType()), T.StructField("surname", T.StringType()),
        T.StructField("dob", T.DateType()), T.StructField("nationality", T.StringType()),
        T.StructField("url", T.StringType())]),
    "lap_times.csv": T.StructType([
        T.StructField("raceId", T.IntegerType()), T.StructField("driverId", T.IntegerType()),
        T.StructField("lap", T.IntegerType()), T.StructField("position", T.IntegerType()),
        T.StructField("time", T.StringType()), T.StructField("milliseconds", T.IntegerType())]),
    "pit_stops.csv": T.StructType([
        T.StructField("raceId", T.IntegerType()), T.StructField("driverId", T.IntegerType()),
        T.StructField("stop", T.IntegerType()), T.StructField("lap", T.IntegerType()),
        T.StructField("time", T.StringType()), T.StructField("duration", T.StringType()),
        T.StructField("milliseconds", T.IntegerType())]),
    "qualifying.csv": T.StructType([
        T.StructField("qualifyId", T.IntegerType()), T.StructField("raceId", T.IntegerType()),
        T.StructField("driverId", T.IntegerType()), T.StructField("constructorId", T.IntegerType()),
        T.StructField("number", T.IntegerType()), T.StructField("position", T.IntegerType()),
        T.StructField("q1", T.StringType()), T.StructField("q2", T.StringType()), T.StructField("q3", T.StringType())]),
    "races.csv": T.StructType([
        T.StructField("raceId", T.IntegerType()), T.StructField("year", T.IntegerType()),
        T.StructField("round", T.IntegerType()), T.StructField("circuitId", T.IntegerType()),
        T.StructField("name", T.StringType()), T.StructField("date", T.DateType()),
        T.StructField("time", T.StringType()), T.StructField("url", T.StringType()),
        T.StructField("fp1_date", T.DateType()), T.StructField("fp1_time", T.StringType()),
        T.StructField("fp2_date", T.DateType()), T.StructField("fp2_time", T.StringType()),
        T.StructField("fp3_date", T.DateType()), T.StructField("fp3_time", T.StringType()),
        T.StructField("quali_date", T.DateType()), T.StructField("quali_time", T.StringType()),
        T.StructField("sprint_date", T.DateType()), T.StructField("sprint_time", T.StringType())]),
    "results.csv": T.StructType([
        T.StructField("resultId", T.IntegerType()), T.StructField("raceId", T.IntegerType()),
        T.StructField("driverId", T.IntegerType()), T.StructField("constructorId", T.IntegerType()),
        T.StructField("number", T.IntegerType()), T.StructField("grid", T.IntegerType()),
        T.StructField("position", T.IntegerType()), T.StructField("positionText", T.StringType()),
        T.StructField("positionOrder", T.IntegerType()), T.StructField("points", T.DoubleType()),
        T.StructField("laps", T.IntegerType()), T.StructField("time", T.StringType()),
        T.StructField("milliseconds", T.IntegerType()), T.StructField("fastestLap", T.IntegerType()),
        T.StructField("rank", T.IntegerType()), T.StructField("fastestLapTime", T.StringType()),
        T.StructField("fastestLapSpeed", T.DoubleType()), T.StructField("statusId", T.IntegerType())]),
    "seasons.csv": T.StructType([T.StructField("year", T.IntegerType()), T.StructField("url", T.StringType())]),
    "sprint_results.csv": T.StructType([
        T.StructField("resultId", T.IntegerType()), T.StructField("raceId", T.IntegerType()),
        T.StructField("driverId", T.IntegerType()), T.StructField("constructorId", T.IntegerType()),
        T.StructField("number", T.IntegerType()), T.StructField("grid", T.IntegerType()),
        T.StructField("position", T.IntegerType()), T.StructField("positionText", T.StringType()),
        T.StructField("positionOrder", T.IntegerType()), T.StructField("points", T.DoubleType()),
        T.StructField("laps", T.IntegerType()), T.StructField("time", T.StringType()),
        T.StructField("milliseconds", T.IntegerType()), T.StructField("fastestLap", T.IntegerType()),
        T.StructField("fastestLapTime", T.StringType()), T.StructField("statusId", T.IntegerType())]),
    "status.csv": T.StructType([T.StructField("statusId", T.IntegerType()), T.StructField("status", T.StringType())])
}

In [9]:
bronze = {file_name.replace(".csv", ""): read_csv_with_schema(file_name, schema)
          for file_name, schema in schemas.items()}

for table_name, df in bronze.items():
    if table_name in {"races", "seasons"}:
        write_parquet(df, f"{BRONZE_PATH}/{table_name}", ["year"])
    elif "raceId" in df.columns:
        write_parquet(df, f"{BRONZE_PATH}/{table_name}", ["raceId"])
    else:
        write_parquet(df, f"{BRONZE_PATH}/{table_name}")

bronze_counts = [(name, df.count()) for name, df in bronze.items()]
spark.createDataFrame(bronze_counts, ["table", "row_count"]).orderBy("table").show(50, truncate=False)

+---------------------+---------+
|table                |row_count|
+---------------------+---------+
|circuits             |77       |
|constructor_results  |12625    |
|constructor_standings|13391    |
|constructors         |212      |
|driver_standings     |34863    |
|drivers              |861      |
|lap_times            |589081   |
|pit_stops            |11371    |
|qualifying           |10494    |
|races                |1125     |
|results              |26759    |
|seasons              |75       |
|sprint_results       |360      |
|status               |139      |
+---------------------+---------+



## Silver Layer - Cleaning, Conformance, and Validation

The Silver layer standardizes columns, handles missing values, normalizes time strings into integer milliseconds, validates joins, and keeps audit columns from ingestion.

In [10]:
def lap_time_to_ms(col_name: str):
    c = F.col(col_name)
    minutes = F.regexp_extract(c, r"^(\\d+):", 1).cast("int")
    seconds = F.regexp_extract(c, r":(\\d+)\\.", 1).cast("int")
    millis = F.regexp_extract(c, r"\\.(\\d+)$", 1).cast("int")
    return F.when(c.rlike(r"^\\d+:\\d{2}\\.\\d{3}$"), minutes * 60000 + seconds * 1000 + millis)

def race_time_to_ms(col_name: str):
    c = F.col(col_name)
    hours = F.regexp_extract(c, r"^(\\d+):\\d{2}:", 1).cast("int")
    minutes = F.regexp_extract(c, r"^\\d+:(\\d{2}):", 1).cast("int")
    seconds = F.regexp_extract(c, r":(\\d{2})\\.", 1).cast("int")
    millis = F.regexp_extract(c, r"\\.(\\d+)$", 1).cast("int")
    return F.when(c.rlike(r"^\\d+:\\d{2}:\\d{2}\\.\\d{3}$"), hours * 3600000 + minutes * 60000 + seconds * 1000 + millis)

def add_missing_flags(df, columns):
    for column in columns:
        df = df.withColumn(f"{column}_was_missing", F.col(column).isNull())
    return df

In [11]:
races = bronze["races"].select("raceId", "year", "round", "circuitId", F.col("name").alias("race_name"), "date", "ingested_at", "source_file")
drivers = bronze["drivers"].withColumn("driver_name", F.concat_ws(" ", "forename", "surname"))
constructors = bronze["constructors"].withColumnRenamed("name", "constructor_name")
circuits = bronze["circuits"].withColumnRenamed("name", "circuit_name")
status = bronze["status"]

results = (
    bronze["results"]
    .transform(lambda df: add_missing_flags(df, ["position", "time", "milliseconds", "fastestLap", "rank", "fastestLapTime", "fastestLapSpeed"]))
    .withColumn("race_duration_ms", F.coalesce(F.col("milliseconds"), race_time_to_ms("time")))
    .withColumn("fastest_lap_ms", lap_time_to_ms("fastestLapTime"))
    .withColumn("position", F.coalesce(F.col("position"), F.col("positionOrder")))
)

qualifying = (
    bronze["qualifying"]
    .transform(lambda df: add_missing_flags(df, ["q1", "q2", "q3"]))
    .withColumn("q1_ms", lap_time_to_ms("q1"))
    .withColumn("q2_ms", lap_time_to_ms("q2"))
    .withColumn("q3_ms", lap_time_to_ms("q3"))
    .withColumn("best_qualifying_ms", F.least("q1_ms", "q2_ms", "q3_ms"))
)

lap_times = (
    bronze["lap_times"]
    .transform(lambda df: add_missing_flags(df, ["time", "milliseconds"]))
    .withColumn("lap_time_ms", F.coalesce(F.col("milliseconds"), lap_time_to_ms("time")))
    .filter(F.col("lap_time_ms").isNotNull())
)

pit_stops = bronze["pit_stops"].withColumnRenamed("milliseconds", "pit_stop_ms")
driver_standings = bronze["driver_standings"]
constructor_standings = bronze["constructor_standings"]

In [12]:
silver = {
    "races": races,
    "drivers": drivers,
    "constructors": constructors,
    "circuits": circuits,
    "status": status,
    "results": results,
    "qualifying": qualifying,
    "lap_times": lap_times,
    "pit_stops": pit_stops,
    "driver_standings": driver_standings,
    "constructor_standings": constructor_standings,
    "constructor_results": bronze["constructor_results"],
    "seasons": bronze["seasons"],
    "sprint_results": bronze["sprint_results"]
}

for table_name, df in silver.items():
    df_to_write = df
    if "raceId" in df.columns and "year" not in df.columns:
        df_to_write = df.join(broadcast(races.select("raceId", "year")), "raceId", "left")
    if "year" in df_to_write.columns:
        write_parquet(df_to_write, f"{SILVER_PATH}/{table_name}", ["year"])
    else:
        write_parquet(df_to_write, f"{SILVER_PATH}/{table_name}")

spark.createDataFrame([(k, v.count()) for k, v in silver.items()], ["table", "row_count"]).orderBy("table").show(50, truncate=False)

+---------------------+---------+
|table                |row_count|
+---------------------+---------+
|circuits             |77       |
|constructor_results  |12625    |
|constructor_standings|13391    |
|constructors         |212      |
|driver_standings     |34863    |
|drivers              |861      |
|lap_times            |589081   |
|pit_stops            |11371    |
|qualifying           |10494    |
|races                |1125     |
|results              |26759    |
|seasons              |75       |
|sprint_results       |360      |
|status               |139      |
+---------------------+---------+



### Referential Integrity Checks

These checks identify orphan keys before Gold tables are produced. In a production pipeline, failed checks could stop the job or write rejected rows to a quarantine area.

In [13]:
integrity_checks = []

integrity_checks.append((
    "results_without_race",
    results.join(races.select("raceId"), "raceId", "left_anti").count()
))
integrity_checks.append((
    "results_without_driver",
    results.join(drivers.select("driverId"), "driverId", "left_anti").count()
))
integrity_checks.append((
    "results_without_constructor",
    results.join(constructors.select("constructorId"), "constructorId", "left_anti").count()
))
integrity_checks.append((
    "qualifying_without_result_key",
    qualifying.join(results.select("raceId", "driverId").distinct(), ["raceId", "driverId"], "left_anti").count()
))

spark.createDataFrame(integrity_checks, ["check", "failed_rows"]).orderBy("check").show(truncate=False)

+-----------------------------+-----------+
|check                        |failed_rows|
+-----------------------------+-----------+
|qualifying_without_result_key|0          |
|results_without_constructor  |0          |
|results_without_driver       |0          |
|results_without_race         |0          |
+-----------------------------+-----------+



## Gold Layer - Analytical Tables

Gold tables are curated for analysis. They combine facts and dimensions into stable business entities that can be used by later notebooks for visualization, machine learning, and graph analytics.

In [14]:
pit_summary = (
    pit_stops.groupBy("raceId", "driverId")
    .agg(
        F.count("stop").alias("pit_stop_count"),
        F.sum("pit_stop_ms").alias("total_pit_stop_ms"),
        F.avg("pit_stop_ms").alias("avg_pit_stop_ms")
    )
)

fact_race_result = (
    results.alias("res")
    .join(broadcast(races).alias("rac"), "raceId", "left")
    .join(qualifying.select("raceId", "driverId", F.col("position").alias("qualifying_position"), "q1_ms", "q2_ms", "q3_ms", "best_qualifying_ms"), ["raceId", "driverId"], "left")
    .join(driver_standings.select("raceId", "driverId", F.col("points").alias("driver_championship_points"), F.col("position").alias("driver_championship_position"), F.col("wins").alias("driver_wins_to_date")), ["raceId", "driverId"], "left")
    .join(constructor_standings.select("raceId", "constructorId", F.col("points").alias("constructor_championship_points"), F.col("position").alias("constructor_championship_position"), F.col("wins").alias("constructor_wins_to_date")), ["raceId", "constructorId"], "left")
    .join(pit_summary, ["raceId", "driverId"], "left")
    .join(broadcast(status.select("statusId", "status")).alias("stat"), "statusId", "left")
    .select(
        "raceId", "year", "round", "race_name", "date", "circuitId",
        "driverId", "constructorId", "grid", "position", "positionOrder", "points", "laps",
        "race_duration_ms", "fastestLap", "fastest_lap_ms", "fastestLapSpeed", "statusId", "status",
        "qualifying_position", "q1_ms", "q2_ms", "q3_ms", "best_qualifying_ms",
        "driver_championship_points", "driver_championship_position", "driver_wins_to_date",
        "constructor_championship_points", "constructor_championship_position", "constructor_wins_to_date",
        F.coalesce("pit_stop_count", F.lit(0)).alias("pit_stop_count"),
        "total_pit_stop_ms", "avg_pit_stop_ms",
        "position_was_missing", "time_was_missing", "milliseconds_was_missing"
    )
)

write_parquet(fact_race_result, f"{GOLD_PATH}/fact_race_result", ["year"])
fact_race_result.orderBy("year", "round", "positionOrder").show(10, truncate=False)

+------+----+-----+------------------+----------+---------+--------+-------------+----+--------+-------------+------+----+----------------+----------+--------------+---------------+--------+--------+-------------------+-----+-----+-----+------------------+--------------------------+----------------------------+-------------------+-------------------------------+---------------------------------+------------------------+--------------+-----------------+---------------+--------------------+----------------+------------------------+
|raceId|year|round|race_name         |date      |circuitId|driverId|constructorId|grid|position|positionOrder|points|laps|race_duration_ms|fastestLap|fastest_lap_ms|fastestLapSpeed|statusId|status  |qualifying_position|q1_ms|q2_ms|q3_ms|best_qualifying_ms|driver_championship_points|driver_championship_position|driver_wins_to_date|constructor_championship_points|constructor_championship_position|constructor_wins_to_date|pit_stop_count|total_pit_stop_ms|avg_pit_

In [15]:
driver_career = (
    results.join(races.select("raceId", "year"), "raceId", "left")
    .groupBy("driverId")
    .agg(F.min("year").alias("career_start_year"), F.max("year").alias("career_end_year"), F.countDistinct("raceId").alias("race_count"))
)

dim_driver = (
    drivers.join(driver_career, "driverId", "left")
    .select(
        "driverId", "driverRef", "driver_name", "number", "code", "dob", "nationality",
        "career_start_year", "career_end_year", "race_count",
        F.lit("1900-01-01").cast("date").alias("effective_start_date"),
        F.lit("9999-12-31").cast("date").alias("effective_end_date"),
        F.lit(True).alias("is_current"),
        "ingested_at", "source_file"
    )
)

write_parquet(dim_driver, f"{GOLD_PATH}/dim_driver")
dim_driver.orderBy("driverId").show(10, truncate=False)

+--------+----------+------------------+------+----+----------+-----------+-----------------+---------------+----------+--------------------+------------------+----------+--------------------------+-----------+
|driverId|driverRef |driver_name       |number|code|dob       |nationality|career_start_year|career_end_year|race_count|effective_start_date|effective_end_date|is_current|ingested_at               |source_file|
+--------+----------+------------------+------+----+----------+-----------+-----------------+---------------+----------+--------------------+------------------+----------+--------------------------+-----------+
|1       |hamilton  |Lewis Hamilton    |44    |HAM |1985-01-07|British    |2007             |2024           |356       |1900-01-01          |9999-12-31        |true      |2026-05-15 19:06:09.406426|drivers.csv|
|2       |heidfeld  |Nick Heidfeld     |NULL  |HEI |1977-05-10|German     |2000             |2011           |184       |1900-01-01          |9999-12-31     

In [16]:
def era_from_year(year_col):
    return (
        F.when(year_col < 1977, F.lit("pre-turbo"))
        .when(year_col.between(1977, 1988), F.lit("turbo"))
        .when(year_col.between(1989, 2005), F.lit("V10"))
        .when(year_col.between(2006, 2013), F.lit("V8"))
        .otherwise(F.lit("hybrid"))
    )

constructor_era = (
    results.join(races.select("raceId", "year"), "raceId", "left")
    .groupBy("constructorId")
    .agg(F.min("year").alias("first_season"), F.max("year").alias("last_season"), F.countDistinct("raceId").alias("race_count"))
)

dim_constructor = (
    constructors.join(constructor_era, "constructorId", "left")
    .withColumn("era_classification", era_from_year(F.col("first_season")))
    .select("constructorId", "constructorRef", "constructor_name", "nationality", "first_season", "last_season", "race_count", "era_classification", "ingested_at", "source_file")
)

write_parquet(dim_constructor, f"{GOLD_PATH}/dim_constructor")
dim_constructor.orderBy("constructorId").show(10, truncate=False)

+-------------+--------------+----------------+-----------+------------+-----------+----------+------------------+------------------------+----------------+
|constructorId|constructorRef|constructor_name|nationality|first_season|last_season|race_count|era_classification|ingested_at             |source_file     |
+-------------+--------------+----------------+-----------+------------+-----------+----------+------------------+------------------------+----------------+
|1            |mclaren       |McLaren         |British    |1968        |2024       |929       |pre-turbo         |2026-05-15 19:06:11.5357|constructors.csv|
|2            |bmw_sauber    |BMW Sauber      |German     |2006        |2009       |70        |V8                |2026-05-15 19:06:11.5357|constructors.csv|
|3            |williams      |Williams        |British    |1975        |2024       |843       |pre-turbo         |2026-05-15 19:06:11.5357|constructors.csv|
|4            |renault       |Renault         |French     

In [17]:
surface_lookup = spark.createDataFrame([
    ("Monaco", "street"), ("Singapore", "street"), ("Azerbaijan", "street"), ("United States", "mixed/permanent"),
    ("United Kingdom", "permanent"), ("Italy", "permanent"), ("Belgium", "permanent")
], ["country", "surface_type"])

dim_circuit = (
    circuits.join(surface_lookup, "country", "left")
    .select(
        "circuitId", "circuitRef", "circuit_name", "location", "country", "lat", "lng", "alt",
        F.coalesce("surface_type", F.lit("unknown / requires enrichment")).alias("surface_type"),
        "ingested_at", "source_file"
    )
)

write_parquet(dim_circuit, f"{GOLD_PATH}/dim_circuit")
dim_circuit.orderBy("country", "circuit_name").show(20, truncate=False)

+---------+-------------+-------------------------------------+----------------+----------+--------+--------+----+-----------------------------+--------------------------+------------+
|circuitId|circuitRef   |circuit_name                         |location        |country   |lat     |lng     |alt |surface_type                 |ingested_at               |source_file |
+---------+-------------+-------------------------------------+----------------+----------+--------+--------+----+-----------------------------+--------------------------+------------+
|25       |galvez       |Autódromo Juan y Oscar Gálvez        |Buenos Aires    |Argentina |-34.6943|-58.4593|8   |unknown / requires enrichment|2026-05-15 19:06:14.145433|circuits.csv|
|29       |adelaide     |Adelaide Street Circuit              |Adelaide        |Australia |-34.9272|138.617 |58  |unknown / requires enrichment|2026-05-15 19:06:14.145433|circuits.csv|
|1        |albert_park  |Albert Park Grand Prix Circuit       |Melbourne   

## SparkSQL Query Examples

The Gold tables are registered as temporary views so analysts can use SQL without changing the physical data model.

In [18]:
fact_race_result.createOrReplaceTempView("fact_race_result")
dim_driver.createOrReplaceTempView("dim_driver")
dim_constructor.createOrReplaceTempView("dim_constructor")
dim_circuit.createOrReplaceTempView("dim_circuit")

spark.sql("""
    SELECT
        d.driver_name,
        COUNT(*) AS race_wins,
        MIN(f.year) AS first_win,
        MAX(f.year) AS latest_win
    FROM fact_race_result f
    JOIN dim_driver d ON f.driverId = d.driverId
    WHERE f.positionOrder = 1
    GROUP BY d.driver_name
    ORDER BY race_wins DESC, latest_win DESC
    LIMIT 10
""").show(truncate=False)

+------------------+---------+---------+----------+
|driver_name       |race_wins|first_win|latest_win|
+------------------+---------+---------+----------+
|Lewis Hamilton    |105      |2007     |2024      |
|Michael Schumacher|91       |1992     |2006      |
|Max Verstappen    |63       |2016     |2024      |
|Sebastian Vettel  |53       |2008     |2019      |
|Alain Prost       |51       |1981     |1993      |
|Ayrton Senna      |41       |1985     |1993      |
|Fernando Alonso   |32       |2003     |2013      |
|Nigel Mansell     |31       |1985     |1994      |
|Jackie Stewart    |27       |1965     |1973      |
|Niki Lauda        |25       |1974     |1985      |
+------------------+---------+---------+----------+



## RDD Demonstration - Fastest Lap per Driver per Race

This section parses `lap_times.csv` as a raw RDD and computes each driver's fastest lap in each race using `mapPartitions` and `reduceByKey`. This demonstrates lower-level Spark programming. For production analytics, the DataFrame implementation below is preferred because Spark can optimize it with Catalyst and Tungsten.

In [19]:
import csv
from io import StringIO

raw_lap_rdd = spark.sparkContext.textFile(f"{RAW_PATH}/lap_times.csv")
header = raw_lap_rdd.first()

def parse_lap_partition(lines):
    for line in lines:
        if line == header:
            continue
        row = next(csv.reader(StringIO(line)))
        race_id, driver_id, lap, position, time_str, milliseconds = row
        if milliseconds and milliseconds != "\\N":
            yield ((int(race_id), int(driver_id)), (int(milliseconds), int(lap), time_str))

fastest_lap_rdd = (
    raw_lap_rdd
    .mapPartitions(parse_lap_partition)
    .reduceByKey(lambda a, b: a if a[0] <= b[0] else b)
)

fastest_lap_rdd_df = fastest_lap_rdd.map(lambda x: (x[0][0], x[0][1], x[1][1], x[1][0], x[1][2])).toDF(
    ["raceId", "driverId", "fastest_lap_number", "fastest_lap_ms", "fastest_lap_time"]
)
fastest_lap_rdd_df.orderBy("raceId", "driverId").show(10, truncate=False)

+------+--------+------------------+--------------+----------------+
|raceId|driverId|fastest_lap_number|fastest_lap_ms|fastest_lap_time|
+------+--------+------------------+--------------+----------------+
|1     |1       |39                |89020         |1:29.020        |
|1     |2       |48                |88283         |1:28.283        |
|1     |3       |48                |87706         |1:27.706        |
|1     |4       |53                |88712         |1:28.712        |
|1     |6       |6                 |89923         |1:29.923        |
|1     |7       |50                |89823         |1:29.823        |
|1     |8       |35                |88488         |1:28.488        |
|1     |9       |36                |87988         |1:27.988        |
|1     |10      |53                |88416         |1:28.416        |
|1     |12      |17                |90502         |1:30.502        |
+------+--------+------------------+--------------+----------------+
only showing top 10 rows


In [20]:
window_fastest_lap = Window.partitionBy("raceId", "driverId").orderBy(F.col("lap_time_ms").asc(), F.col("lap").asc())

fastest_lap_df = (
    lap_times
    .withColumn("rn", F.row_number().over(window_fastest_lap))
    .filter(F.col("rn") == 1)
    .select("raceId", "driverId", F.col("lap").alias("fastest_lap_number"), F.col("lap_time_ms").alias("fastest_lap_ms"), F.col("time").alias("fastest_lap_time"))
)

fastest_lap_df.createOrReplaceTempView("fastest_lap_df")
fastest_lap_df.orderBy("raceId", "driverId").show(10, truncate=False)

+------+--------+------------------+--------------+----------------+
|raceId|driverId|fastest_lap_number|fastest_lap_ms|fastest_lap_time|
+------+--------+------------------+--------------+----------------+
|1     |1       |39                |89020         |1:29.020        |
|1     |2       |48                |88283         |1:28.283        |
|1     |3       |48                |87706         |1:27.706        |
|1     |4       |53                |88712         |1:28.712        |
|1     |6       |6                 |89923         |1:29.923        |
|1     |7       |50                |89823         |1:29.823        |
|1     |8       |35                |88488         |1:28.488        |
|1     |9       |36                |87988         |1:27.988        |
|1     |10      |53                |88416         |1:28.416        |
|1     |12      |17                |90502         |1:30.502        |
+------+--------+------------------+--------------+----------------+
only showing top 10 rows


In [21]:
lap_times.createOrReplaceTempView("lap_times_view")

spark.sql("""
    SELECT raceId, driverId, lap AS fastest_lap_number, lap_time_ms AS fastest_lap_ms, time AS fastest_lap_time
    FROM (
        SELECT
            raceId,
            driverId,
            lap,
            lap_time_ms,
            time,
            ROW_NUMBER() OVER (PARTITION BY raceId, driverId ORDER BY lap_time_ms ASC, lap ASC) AS rn
        FROM lap_times_view
    ) ranked
    WHERE rn = 1
    ORDER BY raceId, driverId
    LIMIT 10
""").show(truncate=False)

+------+--------+------------------+--------------+----------------+
|raceId|driverId|fastest_lap_number|fastest_lap_ms|fastest_lap_time|
+------+--------+------------------+--------------+----------------+
|1     |1       |39                |89020         |1:29.020        |
|1     |2       |48                |88283         |1:28.283        |
|1     |3       |48                |87706         |1:27.706        |
|1     |4       |53                |88712         |1:28.712        |
|1     |6       |6                 |89923         |1:29.923        |
|1     |7       |50                |89823         |1:29.823        |
|1     |8       |35                |88488         |1:28.488        |
|1     |9       |36                |87988         |1:27.988        |
|1     |10      |53                |88416         |1:28.416        |
|1     |12      |17                |90502         |1:30.502        |
+------+--------+------------------+--------------+----------------+



## Output Summary

This notebook creates the following lakehouse outputs:

- Bronze Parquet tables for all 14 raw CSV files.
- Silver Parquet tables with audit columns, normalized time fields, null flags, and validated join keys.
- Gold tables: `fact_race_result`, `dim_driver`, `dim_constructor`, and `dim_circuit`.

The Gold layer is now ready for later phases such as exploratory analytics, visualization, machine learning, graph analysis, and management reporting.